# Practice notebook

Scratch space — nothing here is final, this is just for getting comfortable with pandas on the real VN1 data before writing `load_long()` in `src/data.py`. Feel free to break things and re-run cells.

In [ ]:
import pandas as pd

sales_path = "data/raw/Phase_0_Sales.csv"
price_path = "data/raw/Phase_0_Price.csv"

sales = pd.read_csv(sales_path)
price = pd.read_csv(price_path)

sales.shape, price.shape

((15053, 173), (15053, 173))

## 1. Look around

Both files are wide: one row per (Client, Warehouse, Product), one column per week.

In [2]:
sales.iloc[:5, :6]

,Client,Warehouse,Product,2020-07-06,2020-07-13,2020-07-20
0,0,1,367,7.0,7.0,7.0
1,0,1,639,0.0,0.0,0.0
2,0,1,655,21.0,21.0,21.0
3,0,1,1149,7.0,7.0,7.0
4,0,1,1485,0.0,0.0,0.0


In [3]:
price.iloc[:5, :6]

,Client,Warehouse,Product,2020-07-06,2020-07-13,2020-07-20
0,0,1,367,10.900001,10.900001,10.900001
1,0,1,639,NaN,NaN,NaN
2,0,1,655,21.343332,21.343332,21.343332
3,0,1,1149,11.480000,11.480000,11.480000
4,0,1,1485,NaN,NaN,NaN


## 2. Try melting a small piece

`pd.melt` turns wide (one column per week) into long (one row per series-week). Try it on just the first few rows first, so it's easy to eyeball whether it did what you expect.

In [7]:
key_cols = ["Client", "Warehouse", "Product"]

small = sales
melted = small.melt(id_vars=key_cols, var_name="date", value_name="sales")
melted.head(10)

,Client,Warehouse,Product,date,sales
0,0,1,367,2020-07-06,7.0
1,0,1,639,2020-07-06,0.0
2,0,1,655,2020-07-06,21.0
3,0,1,1149,2020-07-06,7.0
4,0,1,1485,2020-07-06,0.0
5,0,1,1965,2020-07-06,21.0
6,0,1,1969,2020-07-06,0.0
7,0,1,3179,2020-07-06,0.0
8,0,1,3234,2020-07-06,7.0
9,0,1,3463,2020-07-06,0.0


## 3. Poke at the zeros

`CONTEXT.md` says sales are 71.8% zeros. Verify that yourself on the full wide table (excluding the key columns).

In [8]:
value_cols = [c for c in sales.columns if c not in key_cols]
(sales[value_cols] == 0).mean().mean()

np.float64(0.7176357263160362)

## 4. Your turn

Some things worth trying here before writing the real `load_long()`:
- Melt the full `sales` and `price` tables (not just `.head(3)`) and check the row count — should be `15053 * 170`.
- Merge the melted sales and price on `Client, Warehouse, Product, date`.
- Convert `date` to an actual datetime with `pd.to_datetime`.
- Check: is `price` NaN exactly when `sales == 0`, like `CONTEXT.md` claims?

In [3]:
import pandas as pd
df=pd.read_csv("data/raw/Phase_0_Sales.csv")
key =["Client", "Warehouse", "Product"]
df

,Client,Warehouse,Product,2020-07-06,2020-07-13,2020-07-20,2020-07-27,2020-08-03,2020-08-10,2020-08-17,...,2023-07-31,2023-08-07,2023-08-14,2023-08-21,2023-08-28,2023-09-04,2023-09-11,2023-09-18,2023-09-25,2023-10-02
0,0,1,367,7.0,7.0,7.0,7.0,7.0,7.0,7.0,...,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,7.0,0.0
1,0,1,639,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,3.0,5.0,5.0,6.0,5.0,1.0,2.0,2.0,18.0,2.0
2,0,1,655,21.0,21.0,21.0,25.0,35.0,35.0,35.0,...,9.0,4.0,2.0,9.0,8.0,6.0,0.0,17.0,21.0,37.0
3,0,1,1149,7.0,7.0,7.0,7.0,7.0,7.0,7.0,...,1.0,1.0,1.0,0.0,2.0,2.0,2.0,1.0,0.0,1.0
4,0,1,1485,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,21.0,23.0,2.0,1.0,1.0,2.0,0.0,22.0,10.0,6.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15048,46,318,13485,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,38.0,42.0,49.0,0.0,33.0,61.0,67.0,63.0,48.0,80.0
15049,46,318,13582,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,33.0,18.0,20.0,1.0,14.0,41.0,25.0,34.0,30.0,39.0
15050,46,318,13691,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,6.0,1.0,2.0,0.0,0.0,0.0,1.0,4.0,3.0,1.0
15051,46,318,13946,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,36.0,12.0,6.0,0.0,1.0,6.0,6.0,1.0,9.0,3.0


In [4]:
df_sales=df.melt(id_vars=["Client","Warehouse","Product"] , var_name="date", value_name="sales")
df_sales

,Client,Warehouse,Product,date,sales
0,0,1,367,2020-07-06,7.0
1,0,1,639,2020-07-06,0.0
2,0,1,655,2020-07-06,21.0
3,0,1,1149,2020-07-06,7.0
4,0,1,1485,2020-07-06,0.0
...,...,...,...,...,...
2559005,46,318,13485,2023-10-02,80.0
2559006,46,318,13582,2023-10-02,39.0
2559007,46,318,13691,2023-10-02,1.0
2559008,46,318,13946,2023-10-02,3.0


In [6]:
df_sales["date"] = pd.to_datetime(df_sales["date"])
df_sales["date"]

0         2020-07-06
1         2020-07-06
2         2020-07-06
3         2020-07-06
4         2020-07-06
             ...    
2559005   2023-10-02
2559006   2023-10-02
2559007   2023-10-02
2559008   2023-10-02
2559009   2023-10-02
Name: date, Length: 2559010, dtype: datetime64[ns]

In [8]:
df_sales["month"]   = df_sales["date"].dt.month
df_sales["week"]    = df_sales["date"].dt.isocalendar().week.astype(int)
df_sales["quarter"] = df_sales["date"].dt.quarter
df_sales[["date", "month", "week", "quarter"]].head()

,date,month,week,quarter
0,2020-07-06,7,28,3
1,2020-07-06,7,28,3
2,2020-07-06,7,28,3
3,2020-07-06,7,28,3
4,2020-07-06,7,28,3


In [19]:
df_sales=df_sales.sort_values(key+["date"])
df_sales.head(10)

,Client,Warehouse,Product,date,sales,month,week,quarter
0,0,1,367,2020-07-06,7.0,7,28,3
15053,0,1,367,2020-07-13,7.0,7,29,3
30106,0,1,367,2020-07-20,7.0,7,30,3
45159,0,1,367,2020-07-27,7.0,7,31,3
60212,0,1,367,2020-08-03,7.0,8,32,3
75265,0,1,367,2020-08-10,7.0,8,33,3
90318,0,1,367,2020-08-17,7.0,8,34,3
105371,0,1,367,2020-08-24,7.0,8,35,3
120424,0,1,367,2020-08-31,7.0,8,36,3
135477,0,1,367,2020-09-07,7.0,9,37,3


In [ ]:
# --- reset the index after sorting ---
df_sales = df_sales.reset_index(drop=True)
df_sales.head(10)

In [ ]:
# --- PRACTICE — groupby, the workhorse ---
per_series_total = df_sales.groupby(key)["sales"].sum()
per_series_total

In [ ]:
# --- PRACTICE — more groupby aggregations ---
summary = df_sales.groupby(key)["sales"].agg(
    mean_sales="mean",
    zero_rate=lambda s: (s == 0).mean(),
    weeks="count",
)
summary

In [ ]:
# --- PRACTICE — filtering (boolean masks) ---
sold = df_sales[df_sales["sales"] > 0]
print(len(sold), "of", len(df_sales), "rows had a positive sale")
sold.head()

In [ ]:
# --- PRACTICE — a lag feature (core of forecasting) ---
df_sales["sales_last_week"] = df_sales.groupby(key)["sales"].shift(1)
df_sales[key + ["date", "sales", "sales_last_week"]].head(8)
# Notice: the first week of each series is NaN — nothing before it. That's correct.

In [ ]:
# --- PRACTICE — a rolling feature ---
df_sales["sales_roll4"] = df_sales.groupby(key)["sales"].transform(lambda s: s.rolling(4).mean())
df_sales[key + ["date", "sales", "sales_roll4"]].head(8)

In [ ]:
# --- PRACTICE — pivot long back to wide (you'll need this to score) ---
wide_again = df_sales.pivot_table(index=key, columns="date", values="sales")
print(wide_again.shape)           # should match the original sales.shape (minus key cols)
wide_again.iloc[:5, :5]

In [ ]:
# --- PRACTICE — the MA12 baseline, by hand ---
# For each series, forecast = mean of its LAST 12 weeks' sales.
# This is literally the organizers' baseline.
ma12 = wide_again.iloc[:, -12:].mean(axis=1)
ma12.head(10)                     # one forecast value per series